# Prepare raw PBMC data for QC analysis

In [4]:
from pathlib import Path
import pandas as pd
import anndata as ad
from tqdm import tqdm
from fast_array_utils import stats

from scatlastb_utils.io import read_anndata

In [1]:
raw_dir = '~/workspace/immune-v1-scripts/data/large_files/remapped/zarr_files/'

In [2]:
ll {raw_dir}

total 68
drwxrwxr-x 11 michaela.mueller 4096 Aug 20  2024 AIDApilot.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Mar 20  2025 COMBAT2022_raw.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Mar 14  2025 COMBAT2022.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Feb 26  2025 Hao2021.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Aug 20  2024 Lee2020.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Jun  5  2024 Liu2021.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Jun  5  2024 Perez2022.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Apr  8  2025 Powell.zarr/
drwxrwxr-x 11 michaela.mueller 4096 May  5  2025 raw_files_for_better_cell_overlap/
drwxrwxr-x 11 michaela.mueller 4096 Feb 26  2025 SchulteSchrepping2020.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Apr 20  2025 Stephenson_Newcastle.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Mar 14  2025 vanderWijst2021_raw.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Aug 20  2024 vanderWijst2021.zarr/
drwxrwxr-x 11 michaela.mueller 4096 Aug 20  2024 VentoTormo.zarr/
drwxrwxr-x 11 michaela.mu

In [5]:
files = list(Path(raw_dir).expanduser().glob('*.zarr'))

In [6]:
def read_file(file):
    study = file.stem.replace('_raw', '')
    adata = read_anndata(file, dask=True, backed=True, verbose=False)
    adata.obs = adata.obs.set_index('remapping_sample_barcode')
    adata.obs['study'] = study
    return study, adata

In [7]:
adatas = dict(read_file(file) for file in tqdm(files))

  0%|          | 0/16 [00:00<?, ?it/s]/hpc/scratch/project.hca_integration/miniforge3/envs/scanpy/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
  6%|▋         | 1/16 [00:00<00:09,  1.51it/s]/hpc/scratch/project.hca_integration/miniforge3/envs/scanpy/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
 12%|█▎        | 2/16 [00:01<00:10,  1.29it/s]/hpc/scratch/project.hca_integration/miniforge3/envs/scanpy/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
 19%|█▉        | 3/16 [00:01<00:06,  1.97it/s]/hpc/scratch/project.hca_integration/miniforge3/envs/sca

In [8]:
adatas

{'Yoshida2022': AnnData object with n_obs × n_vars = 623630 × 36601
     obs: 'remapping_sample', 'remapping_barcode', 'sequencing_library_stripped', 'study'
     var: 'gene_symbols', 'feature_types',
 'COMBAT2022': AnnData object with n_obs × n_vars = 1149311 × 36601
     obs: 'remapping_sample', 'remapping_barcode', 'matching_barcode', 'study'
     var: 'gene_symbols', 'feature_types',
 'VentoTormo': AnnData object with n_obs × n_vars = 118747 × 36601
     obs: 'remapping_sample', 'remapping_barcode', 'Sample_ID', 'study'
     var: 'gene_symbols', 'feature_types',
 'SchulteSchrepping2020': AnnData object with n_obs × n_vars = 85863 × 36601
     obs: 'remapping_sample', 'remapping_barcode', 'study'
     var: 'gene_symbols', 'feature_types',
 'Villani_normal': AnnData object with n_obs × n_vars = 798214 × 36601
     obs: 'remapping_sample', 'remapping_barcode', 'study'
     var: 'gene_symbols', 'feature_types',
 'vanderWijst2021': AnnData object with n_obs × n_vars = 539916 × 36601
   

In [11]:
{
    study: adata.shape[0] for study, adata in adatas.items()
}

{'Yoshida2022': 623630,
 'COMBAT2022': 1149311,
 'VentoTormo': 118747,
 'SchulteSchrepping2020': 85863,
 'Villani_normal': 798214,
 'vanderWijst2021': 539916,
 'Hao2021': 418382,
 'Powell': 232911,
 'Perez2022': 1989213,
 'Stephenson_Newcastle': 621511,
 'Liu2021': 944571,
 'Lee2020': 86498,
 'AIDApilot': 91664,
 'Yazar2022': 1433674}

In [12]:
sum([adata.shape[0] for adata in adatas.values()])

9134105

In [13]:
# remove completely empty droplets
for study, adata in tqdm(adatas.items()):
    mask = stats.sum(adata.X, axis=1) > 0
    mask = mask.compute()
    adatas[study] = adata[mask].copy()

100%|██████████| 14/14 [00:42<00:00,  3.01s/it]


In [14]:
{
    study: adata.shape[0] for study, adata in adatas.items()
}

{'Yoshida2022': 623630,
 'COMBAT2022': 1149311,
 'VentoTormo': 118747,
 'SchulteSchrepping2020': 85863,
 'Villani_normal': 798214,
 'vanderWijst2021': 539916,
 'Hao2021': 418382,
 'Powell': 232911,
 'Perez2022': 1989213,
 'Stephenson_Newcastle': 621511,
 'Liu2021': 944571,
 'Lee2020': 86498,
 'AIDApilot': 91664,
 'Yazar2022': 1433674}

In [15]:
sum([adata.shape[0] for adata in adatas.values()])

9134105

In [12]:
adata = ad.concat(adatas.values())


KeyboardInterrupt



In [78]:
adata

AnnData object with n_obs × n_vars = 7001268 × 36601
    obs: 'remapping_sample', 'remapping_barcode', 'study'

## Add gene info

In [87]:
gene_anno_file = '~/workspace/immune-v1-scripts/notebooks/remapped_data/gene_symbol_type.tsv'

In [88]:
gene_anno = pd.read_table(
    gene_anno_file,
    header=None,
    names=['feature_name', 'feature_biotype'],
    index_col=0,
)

In [89]:
adata.var = gene_anno

In [90]:
adata

AnnData object with n_obs × n_vars = 7001268 × 36601
    obs: 'remapping_sample', 'remapping_barcode', 'study'
    var: 'feature_name', 'feature_biotype'

In [93]:
adata.obs

,remapping_sample,remapping_barcode,study
remapping_sample_barcode,,,
GSM5140589_AAACCTGAGAAGATTC,GSM5140589,AAACCTGAGAAGATTC,vanderWijst2021
GSM5140589_AAACCTGAGACGCACA,GSM5140589,AAACCTGAGACGCACA,vanderWijst2021
GSM5140589_AAACCTGAGACTAAGT,GSM5140589,AAACCTGAGACTAAGT,vanderWijst2021
GSM5140589_AAACCTGAGAGCCCAA,GSM5140589,AAACCTGAGAGCCCAA,vanderWijst2021
GSM5140589_AAACCTGAGCCATCGC,GSM5140589,AAACCTGAGCCATCGC,vanderWijst2021
...,...,...,...
CV001_KM9294296_TTTGGTTGTGCGATAG,CV001_KM9294296,TTTGGTTGTGCGATAG,Yoshida2022
CV001_KM9294296_TTTGGTTGTTAAGGGC,CV001_KM9294296,TTTGGTTGTTAAGGGC,Yoshida2022
CV001_KM9294296_TTTGTCACAATCCAAC,CV001_KM9294296,TTTGTCACAATCCAAC,Yoshida2022


In [92]:
adata.var

,feature_name,feature_biotype
ENSG00000243485,MIR1302-2HG,lncRNA
ENSG00000237613,FAM138A,lncRNA
ENSG00000186092,OR4F5,protein_coding
ENSG00000238009,AL627309.1,lncRNA
ENSG00000239945,AL627309.3,lncRNA
...,...,...
ENSG00000277836,AC141272.1,protein_coding
ENSG00000278633,AC023491.2,protein_coding
ENSG00000276017,AC007325.1,protein_coding
ENSG00000278817,AC007325.4,protein_coding


## Ingest obs metadata